<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## Verify Iterative Results and Stopping Conditions

**Distinguish approximate stationarity, an exhausted update limit, and a verified global minimum.**

Keep the four clock observations, corrected-error calculation, and squared-error objective. Remove the calibration equality and correction bound; the settings are unrestricted unless the rate is explicitly held at zero. Both settings vary. The objective, starting settings, tolerance, and update limit stay fixed when comparing gradient descent with Newton's method.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 1 · Keep the decision and objective fixed

The observed clock errors are $d=(-2,-1,-1,0)$ min at times $t=(0,1,2,3)$ h. For observation $i=1,\ldots,4$, the corrected error is

$$
y_i=d_i+q+rt_i.
$$

The decision $x=(q,r)$ contains the initial correction $q$ in min and rate correction $r$ in min/h. The response vector is $y=\operatorname{Sim}(x)$. The scalar objective is the sum of squared errors:

$$
f(x)=\sum_{i=1}^{4}(d_i+q+rt_i)^2.
$$

We write $f(x)$ for the objective after composing the response calculation with the squared-error score. Its unit is $\mathrm{min}^2$. Every finite pair of real settings is feasible in this formulation. A smaller score therefore gives a better candidate.

This is a smooth, unconstrained least-squares problem and a quadratic program (QP). These names describe the problem form; gradient descent and Newton's method are solution procedures.

In the calculations, coordinates are numerical values in the stated units: $q=1$ means 1 min, and $r=1$ means 1 min/h. Gradient sizes and step sizes refer to this fixed coordinate convention. Changing units changes the coordinate scaling and can require a different step size.


In [ ]:
import numpy as np

# Fixed observations: each error corresponds to the time at the same position.
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def objective_function(response):
    return float(np.dot(response, response))


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    if decision.shape != (2,) or not np.isfinite(decision).all():
        raise ValueError("Use two finite numerical settings: correction and rate.")
    response = simulate_clock(decision)
    gradient = 2.0 * np.array([response.sum(), OBSERVATION_TIMES @ response])
    return {"decision": decision.copy(), "response": response,
            "feasible": True, "objective": objective_function(response),
            "gradient": gradient,
            "gradient_norm": float(np.linalg.norm(gradient))}


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 2 · Define the two update rules

The superscript $k$ counts numerical updates. Starting from $x^{(0)}=(0,0)$, gradient descent uses

$$
x^{(k+1)}=x^{(k)}-\alpha\nabla f(x^{(k)}),\qquad \alpha=0.05.
$$

For the two-setting calculation, the gradient collects the partial derivatives:

$$
\nabla f(q,r)=\begin{bmatrix}8q+12r-8\\12q+28r-6\end{bmatrix}.
$$

Newton's method uses the Hessian, the matrix of second partial derivatives,

$$
B=\nabla^2 f=\begin{bmatrix}8&12\\12&28\end{bmatrix}.
$$

It solves $Bp^{(k)}=-\nabla f(x^{(k)})$ for a direction $p^{(k)}$ and sets $x^{(k+1)}=x^{(k)}+\eta p^{(k)}$, using a full step $\eta=1$ here. Both functions record the candidate, score, gradient norm, update count, and stopping reason. The shared tolerance is $\varepsilon_g=10^{-6}$ and the update limit is $K=500$.


In [ ]:
def run_gradient_descent(initial_decision=(0.0, 0.0), step_size=0.05,
                         gradient_tolerance=1e-6, max_steps=500):
    if not np.isfinite(step_size) or step_size <= 0:
        raise ValueError("The step size must be positive and finite.")
    if gradient_tolerance < 0 or max_steps < 0:
        raise ValueError("Use a nonnegative tolerance and iteration limit.")
    history = [evaluate_clock(initial_decision)]
    for iteration in range(max_steps + 1):
        current = history[-1]
        if current["gradient_norm"] <= gradient_tolerance:
            reason = "Gradient tolerance reached"
            break
        if iteration == max_steps:
            reason = "Iteration limit reached"
            break
        candidate = current["decision"] - step_size * current["gradient"]
        if not np.isfinite(candidate).all():
            reason = "Nonfinite candidate; update failed"
            break
        result = evaluate_clock(candidate)
        if not np.isfinite(result["objective"]) or not np.isfinite(result["gradient_norm"]):
            reason = "Nonfinite evaluation; update failed"
            break
        history.append(result)
    return {"result": history[-1], "history": history,
            "updates": len(history) - 1, "stop_reason": reason,
            "step_size": step_size, "gradient_tolerance": gradient_tolerance}

In [ ]:
HESSIAN = 2.0 * np.array([
    [len(OBSERVED_ERRORS), OBSERVATION_TIMES.sum()],
    [OBSERVATION_TIMES.sum(), OBSERVATION_TIMES @ OBSERVATION_TIMES],
])


def run_newton(initial_decision=(0.0, 0.0), damping=1.0,
               gradient_tolerance=1e-6, max_steps=500):
    if not np.isfinite(damping) or not 0 < damping <= 1:
        raise ValueError("Use a damping factor in (0, 1].")
    if gradient_tolerance < 0 or max_steps < 0:
        raise ValueError("Use a nonnegative tolerance and iteration limit.")
    history = [evaluate_clock(initial_decision)]
    for iteration in range(max_steps + 1):
        current = history[-1]
        if current["gradient_norm"] <= gradient_tolerance:
            reason = "Gradient tolerance reached"
            break
        if iteration == max_steps:
            reason = "Iteration limit reached"
            break
        direction = np.linalg.solve(HESSIAN, -current["gradient"])
        history.append(evaluate_clock(current["decision"] + damping * direction))
    return {"result": history[-1], "history": history,
            "updates": len(history) - 1, "stop_reason": reason,
            "damping": damping, "gradient_tolerance": gradient_tolerance}

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 3 · Compare work and verify the result

Keep the objective, start $(0,0)$, gradient tolerance $10^{-6}$, and iteration limit 500 identical. Gradient descent uses $\alpha=0.05$; Newton uses $\eta=1$. The plot compares progress in the same score. A full Newton step reaches 0.2 immediately, while gradient descent approaches it over many updates.

The exact reference follows from the stationary equations $8q+12r=8$ and $12q+28r=6$. To verify that their solution is a minimum, let $p=(p_q,p_r)$ be any nonzero change in the settings. Then

$$
p^{\mathsf T}Bp=2\sum_{i=1}^{4}(p_q+t_i p_r)^2>0.
$$

The sum could be zero only if every term were zero. The terms at $t=0$ and $t=1$ would then require both $p_q=0$ and $p_r=0$. Thus $B$ is positive definite. The objective is strictly convex, and $(1.9,-0.6)$ is its unique global minimum. This mathematical structure supplies the guarantee; a short run alone does not.

The vertical axis of the comparison plot is the objective gap $f(x^{(k)})-0.2$ on a logarithmic scale. Zero gaps are displayed at $10^{-14}$ only to make them visible on that scale.

In [ ]:
import sys
import matplotlib

BLUE, TEAL, ORANGE, PURPLE = "#2878b5", "#168578", "#e78b24", "#8854a5"


def get_pyplot(interactive=False):
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry
            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt
    return plt


def show_convergence(gradient_result, newton_result):
    plt = get_pyplot()
    figure, axis = plt.subplots(figsize=(5.6, 3.6))
    for name, run, color in (("Gradient descent", gradient_result, BLUE),
                             ("Newton", newton_result, ORANGE)):
        gaps = [max(item["objective"] - 0.2, 1e-14) for item in run["history"]]
        axis.semilogy(range(len(gaps)), gaps, color=color,
                      marker="o" if name == "Newton" else None, label=name)
    axis.set(xlabel="Number of updates k", ylabel="Objective gap (min²)",
             title="The same score falls at different rates")
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    return figure

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

Both runs satisfy the same gradient test. Their iteration counts measure different amounts of work per update.

In [ ]:
gradient_run = run_gradient_descent()
newton_run = run_newton()
convergence_figure = show_convergence(gradient_run, newton_run)
get_pyplot().show()
for name, run in (("Gradient descent", gradient_run), ("Newton", newton_run)):
    print(f"{name}: {run['updates']} updates, f={run['result']['objective']:.8f} min², "
          f"gradient norm={run['result']['gradient_norm']:.3g}")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

| Method | Information and work in one update | Interpretation here |
|:---|:---|:---|
| Gradient descent | Evaluate the gradient and multiply by a step size | Inexpensive updates; progress depends on step size and scaling |
| Newton | Evaluate curvature and solve a linear system for the direction | One full update solves this positive-definite quadratic |

In a larger problem, forming and solving with a Hessian can cost much more than a gradient update. Fewer updates do not by themselves prove lower runtime. For this two-setting example the linear solve is small, and the constant Hessian can be reused.

The one-step result is special to an exact quadratic with a nonsingular positive-definite Hessian. For a general smooth objective the Hessian changes with the candidate, and the quadratic approximation is only local. A singular Hessian can prevent the Newton solve; an indefinite Hessian can produce a direction that does not decrease the objective. Damping or a line search checks the step size, often together with a modified descent direction. Newton's method does not have an unconditional global guarantee.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 4 · Explain why a run stopped

The norm $\|\nabla f(x)\|_2$ measures the size of the two slopes in the chosen numerical coordinates. A small value indicates approximate stationarity. For this strictly convex clock objective, the exact stationary point is the global minimum. For a general nonconvex objective, a zero gradient may also occur at a maximum or saddle point.

| Observation at termination | Supported conclusion |
|:---|:---|
| $\|\nabla f(x)\|_2\le\varepsilon_g$ | The gradient tolerance is satisfied |
| A very small step or score change | The algorithm moved little; inspect the gradient as well |
| $K$ updates have been used | The computational budget is exhausted |

For example, keep the two-setting start $(0,0)$ but use $\alpha=10^{-12}$. The first step is $(8\times10^{-12},6\times10^{-12})$. Its length is $10^{-11}$, yet the gradient norm stays approximately 10. The tiny change is caused by the hyperparameter, not proximity to a stationary point. A limit of two ordinary updates likewise ends the run before its gradient tolerance is met.

The figure contrasts these three stopping situations using the same clock objective. Only the first satisfies the numerical stationarity test.

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/04-2_iterative_optimization_gradient_descent_and_newton_method/assets/stopping_conditions.svg" alt="Three clock runs distinguish a small gradient from a tiny update and an exhausted budget. The converged run satisfies the gradient tolerance; a step size of 1e-12 barely moves while the gradient norm remains about 10; a two-update budget ends before convergence." width="520" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: white;">
</div>

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

The diagnostic runs explicitly retain their stopping reasons. They do not label a budget-limited candidate as an optimum.

In [ ]:
tiny_run = run_gradient_descent(step_size=1e-12, max_steps=1)
limited_run = run_gradient_descent(max_steps=2)
tiny_step = np.linalg.norm(tiny_run["history"][1]["decision"]
                           - tiny_run["history"][0]["decision"])
print("Tiny step length:", tiny_step)
for name, run in (("Tiny step", tiny_run), ("Two-update budget", limited_run),
                  ("Converged run", gradient_run)):
    print(name, "| gradient norm:", run["result"]["gradient_norm"],
          "|", run["stop_reason"])

<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

### 5 · Source material and scope

The supplied `Lecture_9_2_Traditional_Optimization02/9_2.pptx`, slides 4–8, introduces gradient-based methods, steepest descent, Newton's use of the Hessian, and initialization. Its `Example_9_2/hands_on_1.py` supplies a repeated negative-gradient update, a gradient-norm stopping test, and an iteration limit. This notebook compares the work of the two methods and separates numerical stopping tests from the mathematical minimum guarantee.

The source example's clipping operation is omitted because these settings are unrestricted. Plain gradient and Newton updates do not enforce a calibration equality or correction bound; restoring those requirements would require a constraint-aware method and explicit feasibility checks.
